In [17]:
import os
import librosa
import soundfile as sf
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [ ]:
# 1. Définir les chemins locaux vers ton Google Drive
dossier_source = r'C:\Users\Admin\Desktop\raw'
dossier_destination = r'C:\Users\Admin\Desktop\processed_chunked'


In [ ]:
TARGET_SR = 16000
TARGET_DURATION = 5.0
target_length = int(TARGET_SR * TARGET_DURATION)

In [ ]:
# 2. Lister les fichiers originaux
fichiers_a_traiter = []
print(" Recherche des fichiers dans le dossier source...")

for root, dirs, files in os.walk(dossier_source):
    for file in files:
        if file.endswith('.wav') or file.endswith('.mp3'):
            fichiers_a_traiter.append(os.path.join(root, file))
            # Affichage rassurant pendant la recherche
            if len(fichiers_a_traiter) % 5000 == 0:
                print(f"   ⏳ {len(fichiers_a_traiter)} fichiers trouvés...")

print(f"\nTerminé ! {len(fichiers_a_traiter)} fichiers bruts trouvés.")
print("Début du traitement audio...\n")

 Recherche des fichiers dans le dossier source...
   ⏳ 5000 fichiers trouvés...
   ⏳ 10000 fichiers trouvés...
   ⏳ 15000 fichiers trouvés...
   ⏳ 20000 fichiers trouvés...
   ⏳ 25000 fichiers trouvés...
   ⏳ 30000 fichiers trouvés...
   ⏳ 35000 fichiers trouvés...
   ⏳ 40000 fichiers trouvés...
   ⏳ 45000 fichiers trouvés...
   ⏳ 50000 fichiers trouvés...

Terminé ! 54715 fichiers bruts trouvés.
Début du traitement audio...



In [9]:
# 3. Boucle de découpage avec barre de progression
for filepath in tqdm(fichiers_a_traiter, desc="Découpage & Standardisation (5s)"):
    try:
        # Calcul du chemin de sauvegarde
        chemin_relatif = os.path.relpath(filepath, dossier_source)
        chemin_base_sauvegarde = os.path.join(dossier_destination, chemin_relatif)
        os.makedirs(os.path.dirname(chemin_base_sauvegarde), exist_ok=True)
        nom_fichier_base, _ = os.path.splitext(chemin_base_sauvegarde)
        
        # Si la partie 1 existe déjà, on déduit que le fichier a déjà été traité 
        # et on passe au suivant SANS faire le librosa.load !
        if os.path.exists(f"{nom_fichier_base}_part1.wav"):
            continue
        
        # Chargement de l'audio complet
        y, sr = librosa.load(filepath, sr=TARGET_SR, mono=True)
        
        # Nettoyage des silences aux extrémités
        y_trimmed, _ = librosa.effects.trim(y, top_db=20)
        total_length = len(y_trimmed)
        
        # Ignorer si l'audio est vide (ou ne contenait que du silence)
        if total_length == 0:
            continue
            
        # Calcul du nombre de morceaux de 5s possibles
        num_chunks = int(np.ceil(total_length / target_length))
        
        # Découpage et sauvegarde
        for i in range(num_chunks):
            chemin_sauvegarde_chunk = f"{nom_fichier_base}_part{i+1}.wav"
            
            # SÉCURITÉ : Si le fichier existe déjà, on le saute ! (Anti-coupure)
            if os.path.exists(chemin_sauvegarde_chunk):
                continue
                
            start_sample = i * target_length
            end_sample = start_sample + target_length
            y_chunk = y_trimmed[start_sample:end_sample]
            
            # Padding pour le dernier morceau s'il est trop court
            if len(y_chunk) < target_length:
                padding = target_length - len(y_chunk)
                y_chunk = np.pad(y_chunk, (0, padding), mode='constant')
                
            # Sauvegarde directement sur le Drive via ton PC
            sf.write(chemin_sauvegarde_chunk, y_chunk, TARGET_SR)
            
    except Exception as e:
        print(f"Erreur sur le fichier {filepath} : {e}")

print("\n Travail terminé ! Tes données sont prêtes et partagées sur Drive.")

Découpage & Standardisation (5s): 100%|██████████| 54715/54715 [00:54<00:00, 1008.90it/s]


 Travail terminé ! Tes données sont prêtes et partagées sur Drive.


### L'audio vers l'image

In [10]:
# 1. Définir les dossiers
dossier_source_audio = r'C:\Users\Admin\Desktop\processed_chunked'
dossier_dest_spectro = r'C:\Users\Admin\Desktop\spectrograms'
os.makedirs(dossier_dest_spectro, exist_ok=True)

In [11]:
# 2. Lister tous les morceaux audio de 5s
fichiers_audio = []
print("Recherche des fichiers audio découpés...")
for root, dirs, files in os.walk(dossier_source_audio):
    for file in files:
        if file.endswith('.wav'):
            fichiers_audio.append(os.path.join(root, file))

print(f"{len(fichiers_audio)} audios de 5s prêts pour la transformation.")

Recherche des fichiers audio découpés...
100869 audios de 5s prêts pour la transformation.


In [12]:
# 3. Boucle de génération des matrices
for filepath in tqdm(fichiers_audio, desc="Génération des Spectrogrammes"):
    try:
        # Recréer l'architecture des dossiers (ex: garder les sous-dossiers "standard", "local")
        chemin_relatif = os.path.relpath(filepath, dossier_source_audio)
        chemin_sauvegarde = os.path.join(dossier_dest_spectro, chemin_relatif).replace('.wav', '.npy')

        # Sécurité anti-crash
        if os.path.exists(chemin_sauvegarde):
            continue

        os.makedirs(os.path.dirname(chemin_sauvegarde), exist_ok=True)

        # A. Charger l'audio
        y, sr = librosa.load(filepath, sr=16000, mono=True)

        # B. Calculer le Mel-Spectrogramme
        # n_mels=128 : on divise les fréquences en 128 bandes
        # fmax=8000 : on ignore les sons très aigus (inutiles pour la voix humaine)
        mel_spectrogram = librosa.feature.melspectrogram(y=y, sr=16000, n_mels=128, fmax=8000)
        
        # C. Convertir l'échelle de puissance en Décibels (dB)
        mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)

        # D. Sauvegarder la matrice mathématique
        np.save(chemin_sauvegarde, mel_spectrogram_db)

    except Exception as e:
        print(f"Erreur inattendue sur {filepath} : {e}")

print("\nTous les spectrogrammes sont générés ! Les données sont prêtes pour l'IA.")

Génération des Spectrogrammes: 100%|██████████| 100869/100869 [38:25<00:00, 43.75it/s] 


Tous les spectrogrammes sont générés ! Les données sont prêtes pour l'IA.


In [14]:
dossier_spectro = r'C:\Users\Admin\Desktop\spectrograms'

In [19]:
# 2. Parcourir les dossiers pour créer notre Index
donnees = []
print("Création de l'index des fichiers...")
for root, dirs, files in os.walk(dossier_spectro):
    for file in files:
        if file.endswith('.npy'):
            chemin_complet = os.path.join(root, file)
            
            # Déterminer le label en regardant le nom du dossier parent
            chemin_min = chemin_complet.lower()
            if 'local' in chemin_min:
                label = 0
            elif 'standard' in chemin_min:
                label = 1
            else:
                label = 2
                
            donnees.append({'chemin': chemin_complet, 'label': label})
# Convertir en tableau Pandas pour manipuler ça facilement
df = pd.DataFrame(donnees)
print(f"Index créé ! Total : {len(df)} spectrogrammes trouvés.")

Création de l'index des fichiers...
Index créé ! Total : 100869 spectrogrammes trouvés.


In [20]:
# 3. LE TRIPLE DÉCOUPAGE (TRAIN / VAL / TEST)

# Étape A : On coupe 70% pour le Train, et 30% temporaires (Temp)
df_train, df_temp = train_test_split(df, test_size=0.30, random_state=42, stratify=df['label'])

# Étape B : On coupe les 30% Tempologiques exactement en deux (50% de 30% = 15% chacun)
df_val, df_test = train_test_split(df_temp, test_size=0.50, random_state=42, stratify=df_temp['label'])

# Affichage des résultats
print("\nRépartition finale de tes données :")
print(f"Fichiers Entraînement (Train 70%) : {len(df_train)}")
print(f"Fichiers Validation   (Val 15%)   : {len(df_val)}")
print(f"Fichiers Test final   (Test 15%)  : {len(df_test)}")


Répartition finale de tes données :
Fichiers Entraînement (Train 70%) : 70608
Fichiers Validation   (Val 15%)   : 15130
Fichiers Test final   (Test 15%)  : 15131
